Feature engineering

In [3]:
import pandas as pd
import numpy as np

games = pd.read_csv(
    "nba_clean_full.csv",
    parse_dates=["gameDateTimeEst"]
)

games = games.sort_values(
    "gameDateTimeEst"
).reset_index(drop=True)  # reloading the dataset.

Creating team dataset.

In [7]:
home = games[[
    "gameId",
    "gameDateTimeEst",
    "season_year",
    "home_team",
    "away_team",
    "homeScore",
    "awayScore",
    "home_win",
    "is_playoff"
]].copy()

home = home.rename(columns={
    "home_team": "team",
    "away_team": "opponent",
    "homeScore": "points_for",
    "awayScore": "points_against"
})

home["is_home"] = 1

home["win"] = home["home_win"]
home.head()

,gameId,gameDateTimeEst,season_year,team,opponent,points_for,points_against,home_win,is_playoff,is_home,win
0,22000071,2021-01-01 19:00:00,2021,Dallas Mavericks,Miami Heat,93,83,1,0,1,1
1,22000070,2021-01-01 19:00:00,2021,Detroit Pistons,Boston Celtics,96,93,1,0,1,1
2,22000069,2021-01-01 19:00:00,2021,Charlotte Hornets,Memphis Grizzlies,93,108,0,0,1,0
3,22000072,2021-01-01 19:30:00,2021,Brooklyn Nets,Atlanta Hawks,96,114,0,0,1,0
4,22000074,2021-01-01 20:00:00,2021,Minnesota Timberwolves,Washington Wizards,109,130,0,0,1,0


In [ ]:
away = games[[
    "gameId",
    "gameDateTimeEst",
    "season_year",
    "home_team",
    "away_team",
    "homeScore",
    "awayScore",
    "home_win",
    "is_playoff"
]].copy()

away = away.rename(columns={
    "away_team": "team",
    "home_team": "opponent",
    "awayScore": "points_for",
    "homeScore": "points_against"
})

away["is_home"] = 0

away["win"] = 1 - away["home_win"]
away.head()

,gameId,gameDateTimeEst,season_year,opponent,team,points_against,points_for,home_win,is_playoff,is_home,win
0,22000071,2021-01-01 19:00:00,2021,Dallas Mavericks,Miami Heat,93,83,1,0,0,0
1,22000070,2021-01-01 19:00:00,2021,Detroit Pistons,Boston Celtics,96,93,1,0,0,0
2,22000069,2021-01-01 19:00:00,2021,Charlotte Hornets,Memphis Grizzlies,93,108,0,0,0,1
3,22000072,2021-01-01 19:30:00,2021,Brooklyn Nets,Atlanta Hawks,96,114,0,0,0,1
4,22000074,2021-01-01 20:00:00,2021,Minnesota Timberwolves,Washington Wizards,109,130,0,0,0,1


: 

Here, when we look at them, the home team and the away team's "Team" iD is changed.

Combining both

In [9]:
team_games = pd.concat(
    [home, away],
    ignore_index=True
)

team_games = team_games.sort_values(
    ["team", "gameDateTimeEst"]
).reset_index(drop=True)
team_games.head()

,gameId,gameDateTimeEst,season_year,team,opponent,points_for,points_against,home_win,is_playoff,is_home,win
0,22000072,2021-01-01 19:30:00,2021,Atlanta Hawks,Brooklyn Nets,114,96,0,0,0,1
1,22000083,2021-01-02 19:30:00,2021,Atlanta Hawks,Cleveland Cavaliers,91,96,0,0,1,0
2,22000095,2021-01-04 19:30:00,2021,Atlanta Hawks,New York Knicks,108,113,0,0,1,0
3,22000110,2021-01-06 19:30:00,2021,Atlanta Hawks,Charlotte Hornets,94,102,0,0,1,0
4,22000134,2021-01-09 19:00:00,2021,Atlanta Hawks,Charlotte Hornets,105,113,1,0,0,0


Creating different points

In [10]:
team_games["point_diff"] = (
    team_games["points_for"] -
    team_games["points_against"]
)
team_games.head(20)

,gameId,gameDateTimeEst,season_year,team,opponent,points_for,points_against,home_win,is_playoff,is_home,win,point_diff
0,22000072,2021-01-01 19:30:00,2021,Atlanta Hawks,Brooklyn Nets,114,96,0,0,0,1,18
1,22000083,2021-01-02 19:30:00,2021,Atlanta Hawks,Cleveland Cavaliers,91,96,0,0,1,0,-5
2,22000095,2021-01-04 19:30:00,2021,Atlanta Hawks,New York Knicks,108,113,0,0,1,0,-5
3,22000110,2021-01-06 19:30:00,2021,Atlanta Hawks,Charlotte Hornets,94,102,0,0,1,0,-8
4,22000134,2021-01-09 19:00:00,2021,Atlanta Hawks,Charlotte Hornets,105,113,1,0,0,0,-8
5,22000153,2021-01-11 19:30:00,2021,Atlanta Hawks,Philadelphia 76ers,112,94,1,0,1,1,18
6,22000186,2021-01-15 21:00:00,2021,Atlanta Hawks,Utah Jazz,92,116,1,0,0,0,-24
7,22000195,2021-01-16 22:00:00,2021,Atlanta Hawks,Portland Trail Blazers,106,112,1,0,0,0,-6
8,22000205,2021-01-18 14:30:00,2021,Atlanta Hawks,Minnesota Timberwolves,108,97,1,0,1,1,11
9,22000219,2021-01-20 19:30:00,2021,Atlanta Hawks,Detroit Pistons,123,115,1,0,1,1,8


This will be our main system to see how the team is performing.  - sign represents the home team loosing.

Alright now we will be using rolling features.

In [15]:
team_games["last_5_win_pct"] = (
    team_games # .sort_values(["team", "gameDateTimeEst"]) # already sorted
    .groupby("team")["win"] # group by team
    .transform( # .transform to create a new column
        lambda x: # this is the lambda function that will be applied to each group
        x.shift(1).rolling(5).mean() #.shift(1) to exclude the current game from the calculation and prevent leakage.
    )
)
team_games.head(20) 

,gameId,gameDateTimeEst,season_year,team,opponent,points_for,points_against,home_win,is_playoff,is_home,win,point_diff,last_5_win_pct
0,22000072,2021-01-01 19:30:00,2021,Atlanta Hawks,Brooklyn Nets,114,96,0,0,0,1,18,NaN
1,22000083,2021-01-02 19:30:00,2021,Atlanta Hawks,Cleveland Cavaliers,91,96,0,0,1,0,-5,NaN
2,22000095,2021-01-04 19:30:00,2021,Atlanta Hawks,New York Knicks,108,113,0,0,1,0,-5,NaN
3,22000110,2021-01-06 19:30:00,2021,Atlanta Hawks,Charlotte Hornets,94,102,0,0,1,0,-8,NaN
4,22000134,2021-01-09 19:00:00,2021,Atlanta Hawks,Charlotte Hornets,105,113,1,0,0,0,-8,NaN
5,22000153,2021-01-11 19:30:00,2021,Atlanta Hawks,Philadelphia 76ers,112,94,1,0,1,1,18,0.2
6,22000186,2021-01-15 21:00:00,2021,Atlanta Hawks,Utah Jazz,92,116,1,0,0,0,-24,0.2
7,22000195,2021-01-16 22:00:00,2021,Atlanta Hawks,Portland Trail Blazers,106,112,1,0,0,0,-6,0.2
8,22000205,2021-01-18 14:30:00,2021,Atlanta Hawks,Minnesota Timberwolves,108,97,1,0,1,1,11,0.2
9,22000219,2021-01-20 19:30:00,2021,Atlanta Hawks,Detroit Pistons,123,115,1,0,1,1,8,0.4


So, what the last rolling is doing is ( 0+0+0+1+0 )/5 = 0.2. this is how it is caluclting the win percentage for the last 5 games. we are using rolling 1-5 .s0 the first 5 games and shift(1) is taking the data to one step down.

estimating the strength of the offence and defence.

In [21]:
team_games["offensive average"] = (
    team_games
    .groupby("team")["points_for"]
    .transform(
        lambda x:
        x.shift(1).rolling(5).mean()
    )
)
team_games.head(20)

,gameId,gameDateTimeEst,season_year,team,opponent,points_for,points_against,home_win,is_playoff,is_home,win,point_diff,last_5_win_pct,last_5_points_for,last_5_points_against,Defensive avg,offensive average
0,22000072,2021-01-01 19:30:00,2021,Atlanta Hawks,Brooklyn Nets,114,96,0,0,0,1,18,NaN,NaN,NaN,NaN,NaN
1,22000083,2021-01-02 19:30:00,2021,Atlanta Hawks,Cleveland Cavaliers,91,96,0,0,1,0,-5,NaN,NaN,NaN,NaN,NaN
2,22000095,2021-01-04 19:30:00,2021,Atlanta Hawks,New York Knicks,108,113,0,0,1,0,-5,NaN,NaN,NaN,NaN,NaN
3,22000110,2021-01-06 19:30:00,2021,Atlanta Hawks,Charlotte Hornets,94,102,0,0,1,0,-8,NaN,NaN,NaN,NaN,NaN
4,22000134,2021-01-09 19:00:00,2021,Atlanta Hawks,Charlotte Hornets,105,113,1,0,0,0,-8,NaN,NaN,NaN,NaN,NaN
5,22000153,2021-01-11 19:30:00,2021,Atlanta Hawks,Philadelphia 76ers,112,94,1,0,1,1,18,0.2,102.4,104.0,104.0,102.4
6,22000186,2021-01-15 21:00:00,2021,Atlanta Hawks,Utah Jazz,92,116,1,0,0,0,-24,0.2,102.0,103.6,103.6,102.0
7,22000195,2021-01-16 22:00:00,2021,Atlanta Hawks,Portland Trail Blazers,106,112,1,0,0,0,-6,0.2,102.2,107.6,107.6,102.2
8,22000205,2021-01-18 14:30:00,2021,Atlanta Hawks,Minnesota Timberwolves,108,97,1,0,1,1,11,0.2,101.8,107.4,107.4,101.8
9,22000219,2021-01-20 19:30:00,2021,Atlanta Hawks,Detroit Pistons,123,115,1,0,1,1,8,0.4,104.6,106.4,106.4,104.6


In [25]:
team_games["Defensive avg"] = (
    team_games
    .groupby("team")["points_against"]
    .transform(
        lambda x:
        x.shift(1).rolling(5).mean()
    )
)
team_games.head(20)

,gameId,gameDateTimeEst,season_year,team,opponent,points_for,points_against,home_win,is_playoff,is_home,win,point_diff,last_5_win_pct,last_5_points_for,last_5_points_against,Defensive avg,offensive average,last_5_point_diff
0,22000072,2021-01-01 19:30:00,2021,Atlanta Hawks,Brooklyn Nets,114,96,0,0,0,1,18,NaN,NaN,NaN,NaN,NaN,NaN
1,22000083,2021-01-02 19:30:00,2021,Atlanta Hawks,Cleveland Cavaliers,91,96,0,0,1,0,-5,NaN,NaN,NaN,NaN,NaN,NaN
2,22000095,2021-01-04 19:30:00,2021,Atlanta Hawks,New York Knicks,108,113,0,0,1,0,-5,NaN,NaN,NaN,NaN,NaN,NaN
3,22000110,2021-01-06 19:30:00,2021,Atlanta Hawks,Charlotte Hornets,94,102,0,0,1,0,-8,NaN,NaN,NaN,NaN,NaN,NaN
4,22000134,2021-01-09 19:00:00,2021,Atlanta Hawks,Charlotte Hornets,105,113,1,0,0,0,-8,NaN,NaN,NaN,NaN,NaN,NaN
5,22000153,2021-01-11 19:30:00,2021,Atlanta Hawks,Philadelphia 76ers,112,94,1,0,1,1,18,0.2,102.4,104.0,104.0,102.4,-1.6
6,22000186,2021-01-15 21:00:00,2021,Atlanta Hawks,Utah Jazz,92,116,1,0,0,0,-24,0.2,102.0,103.6,103.6,102.0,-1.6
7,22000195,2021-01-16 22:00:00,2021,Atlanta Hawks,Portland Trail Blazers,106,112,1,0,0,0,-6,0.2,102.2,107.6,107.6,102.2,-5.4
8,22000205,2021-01-18 14:30:00,2021,Atlanta Hawks,Minnesota Timberwolves,108,97,1,0,1,1,11,0.2,101.8,107.4,107.4,101.8,-5.6
9,22000219,2021-01-20 19:30:00,2021,Atlanta Hawks,Detroit Pistons,123,115,1,0,1,1,8,0.4,104.6,106.4,106.4,104.6,-1.8


This will now take the averages from last 5 games and then make the offensive score. for ege. the team has scored - amount of points against an opponent takes the averages of that and then divides it by 5 to make the offensive average. its oppoisit to the defensive average. It looks at how much points have been conceded. The lower the defensive average the better.

Overall team strength is calculated by taking the difference between the offensive and defensive average. The higher the overall strength, the better the team is performing.

In [24]:
team_games["last_5_point_diff"] = (
    team_games
    .groupby("team")["point_diff"]
    .transform(
        lambda x:
        x.shift(1).rolling(5).mean()
    )
)
team_games.head(20)

,gameId,gameDateTimeEst,season_year,team,opponent,points_for,points_against,home_win,is_playoff,is_home,win,point_diff,last_5_win_pct,last_5_points_for,last_5_points_against,Defensive avg,offensive average,last_5_point_diff
0,22000072,2021-01-01 19:30:00,2021,Atlanta Hawks,Brooklyn Nets,114,96,0,0,0,1,18,NaN,NaN,NaN,NaN,NaN,NaN
1,22000083,2021-01-02 19:30:00,2021,Atlanta Hawks,Cleveland Cavaliers,91,96,0,0,1,0,-5,NaN,NaN,NaN,NaN,NaN,NaN
2,22000095,2021-01-04 19:30:00,2021,Atlanta Hawks,New York Knicks,108,113,0,0,1,0,-5,NaN,NaN,NaN,NaN,NaN,NaN
3,22000110,2021-01-06 19:30:00,2021,Atlanta Hawks,Charlotte Hornets,94,102,0,0,1,0,-8,NaN,NaN,NaN,NaN,NaN,NaN
4,22000134,2021-01-09 19:00:00,2021,Atlanta Hawks,Charlotte Hornets,105,113,1,0,0,0,-8,NaN,NaN,NaN,NaN,NaN,NaN
5,22000153,2021-01-11 19:30:00,2021,Atlanta Hawks,Philadelphia 76ers,112,94,1,0,1,1,18,0.2,102.4,104.0,104.0,102.4,-1.6
6,22000186,2021-01-15 21:00:00,2021,Atlanta Hawks,Utah Jazz,92,116,1,0,0,0,-24,0.2,102.0,103.6,103.6,102.0,-1.6
7,22000195,2021-01-16 22:00:00,2021,Atlanta Hawks,Portland Trail Blazers,106,112,1,0,0,0,-6,0.2,102.2,107.6,107.6,102.2,-5.4
8,22000205,2021-01-18 14:30:00,2021,Atlanta Hawks,Minnesota Timberwolves,108,97,1,0,1,1,11,0.2,101.8,107.4,107.4,101.8,-5.6
9,22000219,2021-01-20 19:30:00,2021,Atlanta Hawks,Detroit Pistons,123,115,1,0,1,1,8,0.4,104.6,106.4,106.4,104.6,-1.8


This is the overall strength of the team. The higher the overall strength, the better the team is performing. Atlanta hawks is bad. 